# Assignment 9 - VAE for Compression and Denoising
Run: Runtime → Change runtime type → **T4 GPU**, then Runtime → **Run all** (takes about 3-5 minutes).

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, torchvision, torchvision.transforms as T
import matplotlib.pyplot as plt, pandas as pd

dev = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", dev)
tr = torchvision.datasets.MNIST("./data", train=True, download=True, transform=T.ToTensor())
te = torchvision.datasets.MNIST("./data", train=False, download=True, transform=T.ToTensor())
loader = torch.utils.data.DataLoader(tr, batch_size=128, shuffle=True)

class VAE(nn.Module):
    def __init__(s, z):
        super().__init__()
        s.fc1 = nn.Linear(784, 512); s.mu = nn.Linear(512, z); s.lv = nn.Linear(512, z)
        s.fc3 = nn.Linear(z, 512);   s.fc4 = nn.Linear(512, 784)
    def encode(s, x):
        h = F.relu(s.fc1(x.view(-1, 784))); return s.mu(h), s.lv(h)
    def decode(s, z): return torch.sigmoid(s.fc4(F.relu(s.fc3(z))))
    def forward(s, x):
        mu, lv = s.encode(x)
        z = mu + torch.randn_like(mu) * torch.exp(0.5 * lv)
        return s.decode(z), mu, lv

def train_vae(z, epochs=10):
    torch.manual_seed(0)
    m = VAE(z).to(dev); opt = torch.optim.Adam(m.parameters(), 1e-3)
    for ep in range(1, epochs + 1):
        tot = 0
        for x, _ in loader:
            x = x.to(dev)
            rec, mu, lv = m(x)
            loss = F.binary_cross_entropy(rec, x.view(-1, 784), reduction="sum") \
                   - 0.5 * torch.sum(1 + lv - mu.pow(2) - lv.exp())
            opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item()
        print(f"[latent={z}] epoch {ep}/{epochs} loss/img = {tot/len(tr):.2f}")
    return m.eval()

## Train the VAE with latent dimension 2, 16 and 32

In [ ]:
DIMS = [2, 16, 32]
models = {z: train_vae(z) for z in DIMS}

In [ ]:
# Test data: 10 test images + Gaussian noise (same noise for every model)
x10 = torch.stack([te[i][0] for i in range(10)]).to(dev)
torch.manual_seed(1)
SIGMA = 0.3
noisy = (x10 + SIGMA * torch.randn_like(x10)).clamp(0, 1)
x_all = (te.data.float() / 255).view(-1, 1, 28, 28).to(dev)

@torch.no_grad()
def recon(m, x):
    mu, _ = m.encode(x); return m.decode(mu).view(-1, 1, 28, 28)

def row(ax_row, imgs):
    for a, im in zip(ax_row, imgs):
        a.imshow(im.squeeze().cpu(), cmap="gray"); a.axis("off")

def grid(rows, names, fname, title):
    fig, ax = plt.subplots(len(rows), 10, figsize=(14, 1.5 * len(rows) + 0.6))
    for r, (imgs, n) in enumerate(zip(rows, names)):
        row(ax[r], imgs); ax[r][0].set_title(n, fontsize=9, loc="left")
    fig.suptitle(title); plt.tight_layout(); plt.savefig(fname, dpi=120); plt.show()

## Part A - Compression

In [ ]:
for z in DIMS:
    print(f"Latent dim {z:>2}: original = 784 values, latent = {z} values, compression ratio = {784/z:.1f} : 1")

## Part B - Reconstruction (10 test images, MSE for each)

In [ ]:
for z in DIMS:
    r = recon(models[z], x10)
    mse = ((r - x10) ** 2).mean(dim=(1, 2, 3))
    print(f"\nLatent {z}: per-image MSE =", [round(v, 4) for v in mse.tolist()], "| mean =", round(mse.mean().item(), 4))
    grid([x10, r], ["Original", f"Reconstruction (latent={z})"], f"recon_z{z}.png", f"Part B - Reconstruction, latent dim = {z}")

## Part C - Denoising (Gaussian noise, sigma = 0.3)

In [ ]:
for z in DIMS:
    d = recon(models[z], noisy)
    print(f"Latent {z}: MSE(noisy vs clean) = {((noisy-x10)**2).mean().item():.4f} | MSE(denoised vs clean) = {((d-x10)**2).mean().item():.4f}")
    grid([x10, noisy, d], ["Original clean", "Noisy", f"Denoised (latent={z})"], f"denoise_z{z}.png", f"Part C - Denoising, latent dim = {z}")

## Part D - Results table (changing the latent dimension)

In [ ]:
rows = []
for z in DIMS:
    m = models[z]
    rows.append({
        "Latent dim": z,
        "Compression ratio": f"{784/z:.1f}:1",
        "Recon MSE (10 imgs)": round(((recon(m, x10) - x10) ** 2).mean().item(), 4),
        "Recon MSE (10k test)": round(((recon(m, x_all) - x_all) ** 2).mean().item(), 4),
        "MSE noisy vs clean": round(((noisy - x10) ** 2).mean().item(), 4),
        "MSE denoised vs clean": round(((recon(m, noisy) - x10) ** 2).mean().item(), 4),
    })
df = pd.DataFrame(rows); df.to_csv("results_table.csv", index=False)
display(df)

# One combined comparison image: original, then reconstruction for each latent size
grid([x10] + [recon(models[z], x10) for z in DIMS], ["Original"] + [f"Latent {z}" for z in DIMS],
     "compare_latent_dims.png", "Part D - Reconstruction vs latent dimension")

In [ ]:
# Optional: download all images and the table
from google.colab import files
import glob
for f in glob.glob("*.png") + ["results_table.csv"]:
    files.download(f)